In [1]:
import pandas as pd

# Define the filenames
odisha_filename = 'odisha_weather_crop_data.csv'
soil_filename = 'data_core.csv'

print("--- Checking for unique crop names in both datasets ---")

try:
    # --- Load and process the Odisha dataset ---
    odisha_df = pd.read_csv(odisha_filename)
    # Clean column names by stripping whitespace
    odisha_df.columns = odisha_df.columns.str.strip()
    # Get unique crop names, convert to lowercase, and sort them
    odisha_crops = sorted(odisha_df['Crop'].str.lower().unique())
    
    print(f"\nFound {len(odisha_crops)} unique crop names in '{odisha_filename}':")
    print(odisha_crops)

except FileNotFoundError:
    print(f"\nError: Could not find the file '{odisha_filename}'. Please make sure it's in the same directory.")
except Exception as e:
    print(f"\nAn error occurred while reading '{odisha_filename}': {e}")


try:
    # --- Load and process the Soil dataset ---
    soil_df = pd.read_csv(soil_filename)
    # Clean column names by stripping whitespace
    soil_df.columns = soil_df.columns.str.strip()
    # Get unique crop names, convert to lowercase, and sort them
    soil_crops = sorted(soil_df['Crop Type'].str.lower().unique())

    print(f"\nFound {len(soil_crops)} unique crop names in '{soil_filename}':")
    print(soil_crops)

except FileNotFoundError:
    print(f"\nError: Could not find the file '{soil_filename}'. Please make sure it's in the same directory.")
except Exception as e:
    print(f"\nAn error occurred while reading '{soil_filename}': {e}")

--- Checking for unique crop names in both datasets ---

Found 40 unique crop names in 'odisha_weather_crop_data.csv':
['arhar/tur', 'bajra', 'castor seed', 'coriander', 'cotton(lint)', 'cowpea(lobia)', 'dry chillies', 'garlic', 'ginger', 'gram', 'groundnut', 'horse-gram', 'jowar', 'jute', 'linseed', 'maize', 'masoor', 'mesta', 'moong(green gram)', 'niger seed', 'onion', 'other kharif pulses', 'other rabi pulses', 'peas & beans (pulses)', 'potato', 'ragi', 'rapeseed &mustard', 'rice', 'safflower', 'sannhamp', 'sesamum', 'small millets', 'soyabean', 'sugarcane', 'sunflower', 'sweet potato', 'tobacco', 'turmeric', 'urad', 'wheat']

Found 11 unique crop names in 'data_core.csv':
['barley', 'cotton', 'ground nuts', 'maize', 'millets', 'oil seeds', 'paddy', 'pulses', 'sugarcane', 'tobacco', 'wheat']


In [2]:
import pandas as pd

# --- Step 1: Define the Crop Name Mapping ---
# This dictionary maps the specific names from your Odisha dataset (the key)
# to the general categories in the data_core.csv file (the value).
crop_mapping = {
    'arhar/tur': 'pulses',
    'bajra': 'millets',
    'castor seed': 'oil seeds',
    'coriander': 'pulses', # Often grown with pulses
    'cotton(lint)': 'cotton',
    'cowpea(lobia)': 'pulses',
    'gram': 'pulses',
    'groundnut': 'ground nuts',
    'horse-gram': 'pulses',
    'jowar': 'millets',
    'linseed': 'oil seeds',
    'maize': 'maize',
    'masoor': 'pulses',
    'moong(green gram)': 'pulses',
    'niger seed': 'oil seeds',
    'other kharif pulses': 'pulses',
    'other rabi pulses': 'pulses',
    'peas & beans (pulses)': 'pulses',
    'ragi': 'millets',
    'rapeseed &mustard': 'oil seeds',
    'rice': 'paddy', # Standardizing to 'paddy' which is in data_core
    'safflower': 'oil seeds',
    'sesamum': 'oil seeds',
    'small millets': 'millets',
    'soyabean': 'oil seeds',
    'sugarcane': 'sugarcane',
    'sunflower': 'oil seeds',
    'tobacco': 'tobacco',
    'urad': 'pulses',
    'wheat': 'wheat'
}

# --- Step 2: Load Both Datasets ---
print("Loading datasets...")
odisha_df = pd.read_csv('odisha_weather_crop_data.csv')
soil_df = pd.read_csv('data_core.csv')
# Clean column names just in case
odisha_df.columns = odisha_df.columns.str.strip()
soil_df.columns = soil_df.columns.str.strip()
print("Datasets loaded successfully.")

# --- Step 3: Calculate Average Nutrients per Crop Category ---
print("Calculating average soil nutrient requirements per crop...")
soil_df.rename(columns={'Crop Type': 'Crop_Category'}, inplace=True)
avg_nutrients = soil_df.groupby('Crop_Category')[['Nitrogen', 'Potassium', 'Phosphorous']].mean().reset_index()

# --- Step 4: Merge Datasets Using the Mapping ---
print("\nStandardizing crop names and merging datasets...")
# Create a new column with the standardized category name
odisha_df['Crop_Category'] = odisha_df['Crop'].str.lower().map(crop_mapping)

# Prepare the soil dataset for merging (use lowercase for consistency)
avg_nutrients['Crop_Category'] = avg_nutrients['Crop_Category'].str.lower()

# Merge the two dataframes based on the new 'Crop_Category' column
final_df = pd.merge(odisha_df, avg_nutrients, on='Crop_Category', how='left')

# Drop rows where the merge failed (i.e., crops we didn't map)
final_df.dropna(subset=['Nitrogen', 'Potassium', 'Phosphorous'], inplace=True)
print("Merge complete.")

# --- Step 5: Save the Final, Enriched Dataset ---
final_filename = 'odisha_final_data.csv'
final_df.to_csv(final_filename, index=False)

print(f"\n✅ Success! Final, fully-enriched dataset saved as '{final_filename}'.")
print("It now includes weather data AND representative soil nutrient data.")
print("\nFirst 5 rows of the final dataset:")
print(final_df.head())

Loading datasets...
Datasets loaded successfully.
Calculating average soil nutrient requirements per crop...

Standardizing crop names and merging datasets...
Merge complete.

✅ Success! Final, fully-enriched dataset saved as 'odisha_final_data.csv'.
It now includes weather data AND representative soil nutrient data.

First 5 rows of the final dataset:
    State  District       Crop     Year  Season     Area Area Units  \
0  Odisha    ANUGUL  Arhar/Tur  2002-03  Kharif   8730.0    Hectare   
1  Odisha    ANUGUL  Arhar/Tur  2003-04  Kharif   9500.0    Hectare   
2  Odisha  BALANGIR  Arhar/Tur  2001-02  Kharif  10010.0    Hectare   
3  Odisha  BALANGIR  Arhar/Tur  2002-03  Kharif   7680.0    Hectare   
4  Odisha  BALANGIR  Arhar/Tur  2003-04  Kharif   9160.0    Hectare   

   Production Production Units     Yield   Avg_Temp  Total_Precipitation  \
0      6050.0           Tonnes  0.693013  27.538856              1014.46   
1      6500.0           Tonnes  0.684211  27.802451              1